In [35]:
import pathlib
from chronos import BaseChronosPipeline, Chronos2Pipeline
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# ignore threadpoolctl warnings from pymc
import warnings
warnings.filterwarnings('ignore', module="threadpoolctl")

In [36]:
train_folder = pathlib.Path("..") / "assets" / "CBM_dataset" / "train"

train_dataframes = {}
for csv_file in train_folder.glob("*.csv"):
    csv_file_data = pd.read_csv(csv_file)
    csv_file_data.drop_duplicates(subset=["machine_id", "time"], inplace=True)
    train_dataframes[csv_file.stem] = csv_file_data

list(train_dataframes.keys())

['complete_df_train']

In [37]:
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
for name, df in train_dataframes.items():
	print(f"{name}:")
	print(df.head())
	print("-" * 120)

complete_df_train:
   machine_id  time  degra_level_observed  region  route      speed      load  brake_temperature  vibration_level  rul  segment_id
0           1     0              0.000000       1      1  33.752256  0.718434           0.000000         0.047835   88           1
1           1     1              0.042079       1      0  90.884743  0.637161           3.031550         0.000000   87           1
2           1     2              0.000000       1      1  29.915994  0.539073           0.000000         0.142555   86           1
3           1     3              0.000000       1      1  33.888960  0.630980           0.000000         0.000000   85           1
4           1     4              0.000000       1      1  32.337547  0.538448           7.716752         0.059371   84           1
------------------------------------------------------------------------------------------------------------------------


In [38]:
fig = make_subplots(rows=1, cols=2)

fig.add_trace(go.Scatter(x=df['degra_level_observed'], y=df['brake_temperature'], mode='markers', marker=dict(opacity=0.5), name='Brake Temp'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['degra_level_observed'], y=df['vibration_level'], mode='markers', marker=dict(opacity=0.5), name='Vibration'), row=1, col=2)

fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=1)
fig.update_yaxes(title_text='Brake Temperature', row=1, col=1)
fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=2)
fig.update_yaxes(title_text='Vibration Level', row=1, col=2)

fig.update_layout(
    title='Brake Temperature and Vibration Level vs Degradation Level',
    width=1000, height=450,
)
fig.show()

In [39]:
fig = make_subplots(rows=1, cols=2)

fig.add_trace(go.Scatter(x=df['degra_level_observed'], y=df['speed'], mode='markers', marker=dict(opacity=0.5), name='Speed'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['degra_level_observed'], y=df['load'], mode='markers', marker=dict(opacity=0.5), name='Load'), row=1, col=2)

fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=1)
fig.update_yaxes(title_text='Speed', row=1, col=1)
fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=2)
fig.update_yaxes(title_text='Load', row=1, col=2)

fig.update_layout(
    title='Speed and Load vs Degradation Level',
    width=1000, height=450,
)
fig.show()

In [40]:
fig = make_subplots(rows=1, cols=2)

fig.add_trace(go.Histogram2d(x=df['degra_level_observed'], y=df['speed'], nbinsx=30, nbinsy=30, colorscale='Blues', name='Speed'), row=1, col=1)
fig.add_trace(go.Histogram2d(x=df['degra_level_observed'], y=df['load'], nbinsx=30, nbinsy=30, colorscale='Blues', name='Load'), row=1, col=2)

fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=1)
fig.update_yaxes(title_text='Speed', row=1, col=1)
fig.update_xaxes(title_text='Degradation Level Observed', row=1, col=2)
fig.update_yaxes(title_text='Load', row=1, col=2)

fig.update_layout(
    title='Speed and Load vs Degradation Level (2D Histogram)',
    width=1000, height=450,
)
fig.show()

In [41]:
fig = px.scatter(df, x='degra_level_observed', y='rul', opacity=0.5,
                 labels={'degra_level_observed': 'Degradation Level Observed', 'rul': 'RUL (Remaining Useful Life)'},
                 title='RUL vs Degradation Level Observed')
fig.show()

In [42]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Degradation Level as a Function of Time', 'RUL as a Function of Time'))

for machine_id, group in df.groupby('machine_id'):
    fig.add_trace(go.Scatter(x=group['time'], y=group['degra_level_observed'], mode='lines',
                             opacity=0.6, name=f'Machine {machine_id}', legendgroup=f'machine_{machine_id}'),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=group['time'], y=group['rul'], mode='lines',
                             opacity=0.6, name=f'Machine {machine_id}', legendgroup=f'machine_{machine_id}', showlegend=False),
                  row=2, col=1)

fig.update_yaxes(title_text='Degradation Level Observed', row=1, col=1)
fig.update_yaxes(title_text='RUL (Remaining Useful Life)', row=2, col=1)
fig.update_xaxes(title_text='Time', row=2, col=1)
fig.update_layout(width=1000, height=700)
fig.show()

In [43]:
plot_df = df.sort_values(["machine_id", "time"]).copy()
plot_df["degra_derivative"] = (
    plot_df.groupby("machine_id")["degra_level_observed"].diff()
    / plot_df.groupby("machine_id")["time"].diff()
).fillna(0.0)

plot_df["speed_x_load"] = plot_df["speed"] * plot_df["load"]

fig = px.scatter(plot_df, x='speed_x_load', y='degra_derivative', opacity=0.35,
                 labels={'speed_x_load': 'Speed × Load', 'degra_derivative': 'Degradation Level Derivative'},
                 title='Degradation Level Derivative vs Speed × Load')
fig.show()

In [44]:
fig = px.scatter(plot_df, x='degra_level_observed', y='degra_derivative', opacity=0.35,
                 labels={'degra_level_observed': 'Degradation Level Observed', 'degra_derivative': 'Degradation Level Derivative'},
                 title='Degradation Level Derivative vs Degradation Level')
fig.show()

In [45]:
sorted(df['region'].unique())

[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

In [46]:
regions = sorted(df['region'].unique())
fig = make_subplots(rows=1, cols=len(regions), subplot_titles=[f'Region {r}' for r in regions], shared_xaxes=True)

for idx, region in enumerate(regions, start=1):
    region_data = plot_df[plot_df['region'] == region]['degra_derivative']
    region_data = region_data[np.isfinite(region_data)]
    fig.add_trace(go.Histogram(x=region_data, nbinsx=50, opacity=0.7, name=f'Region {region}'), row=1, col=idx)
    fig.update_xaxes(title_text='Degradation Derivative', row=1, col=idx)
    if idx == 1:
        fig.update_yaxes(title_text='Frequency', row=1, col=1)

fig.update_layout(title='Degradation Derivative by Region', width=1200, height=400, showlegend=False)
fig.show()

In [47]:
routes = sorted(df['route'].unique())
fig = make_subplots(rows=1, cols=len(routes), subplot_titles=[f'Route {r}' for r in routes], shared_xaxes=True)

for idx, route in enumerate(routes, start=1):
    route_data = plot_df[plot_df['route'] == route]['degra_derivative']
    route_data = route_data[np.isfinite(route_data)]
    fig.add_trace(go.Histogram(x=route_data, nbinsx=50, opacity=0.7, name=f'Route {route}'), row=1, col=idx)
    fig.update_xaxes(title_text='Degradation Derivative', row=1, col=idx)
    if idx == 1:
        fig.update_yaxes(title_text='Frequency', row=1, col=1)

fig.update_layout(title='Degradation Derivative by Route', width=1200, height=400, showlegend=False)
fig.show()

In [48]:
train_data = train_dataframes["complete_df_train"].drop(columns=["rul", "segment_id"])

train_data.rename(columns={"machine_id": "item_id", "time": "timestamp", "degra_level_observed": "target"}, inplace=True)

train_data["timestamp"] = pd.to_datetime(train_data["timestamp"], unit="D")

split_train_data = []
for item_id in train_data["item_id"].unique():
    machine_data = train_data[train_data["item_id"] == item_id].copy()
    machine_data["change_detected"] = machine_data["target"].diff().fillna(0).abs() > 8  # threshold for change detection
    machine_data["item_id"] = machine_data["item_id"].astype(str) + "_" + machine_data["change_detected"].cumsum().astype(str)

    split_train_data.append(machine_data.drop(columns=["change_detected"]))

split_train_data = pd.concat(split_train_data)

In [49]:
prediction_length = 10

In [50]:
ready_for_prediction_data = []
for item_id in split_train_data["item_id"].unique():
    machine_data = split_train_data[split_train_data["item_id"] == item_id].iloc[:-prediction_length].copy()

    if len(machine_data) > prediction_length:
        ready_for_prediction_data.append(machine_data)
    else:
        raise ValueError(f"Machine {item_id} does not have enough data for prediction after splitting. Required: >{prediction_length}, Available: {len(machine_data)}")


split_train_data_for_prediction = pd.concat(ready_for_prediction_data)

In [51]:
pipeline: Chronos2Pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# Generate predictions with covariates
train_pred_df = pipeline.predict_df(
    split_train_data_for_prediction,
    prediction_length=prediction_length,
)

merged_df = split_train_data.merge(train_pred_df, on=["item_id", "timestamp"], how="inner")

# Compute weighted quantile loss
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
quantile_cols = [str(q) for q in quantiles]

# Calculate quantile loss for each quantile
quantile_losses = {}
for q, col in zip(quantiles, quantile_cols):
    # Quantile loss: (q - 1) * (y - pred) if y < pred else q * (y - pred)
    errors = merged_df['target'] - merged_df[col]
    loss = np.where(errors >= 0, q * errors, (q - 1) * errors).mean()
    quantile_losses[col] = loss

# Calculate overall weighted quantile loss
weighted_loss = np.mean(list(quantile_losses.values()))

# Save results
zero_shot_results = {
    'quantile_losses': quantile_losses,
    'weighted_quantile_loss': weighted_loss
}

In [52]:
# Prepare data for visualization
fig = go.Figure()

# Get unique item_ids
item_ids = train_data["item_id"].unique()

for item_id in item_ids:
    show_legend = True
    # Get ground truth data
    truth_data = train_data[train_data["item_id"] == item_id]

    if len(truth_data) > 0:
        # Add ground truth line
        fig.add_trace(go.Scatter(
            x=truth_data["timestamp"],
            y=truth_data["target"],
            mode='lines',
            name=f"Truth {item_id}",
            visible=True,
            legendgroup=str(item_id),
            hovertemplate="<b>Ground Truth</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>"
        ))

    
    # Get prediction data
    pred_item_ids = train_pred_df["item_id"].unique()
    for pred_item_id in pred_item_ids:
        if pred_item_id.startswith(str(item_id) + "_"):
            pred_data = train_pred_df[train_pred_df["item_id"] == pred_item_id]
    
        if len(pred_data) > 0:
        
            # Add prediction line
            fig.add_trace(go.Scatter(
                x=pred_data["timestamp"],
                y=pred_data["predictions"],
                mode='lines',
                name=f"Predictions {item_id}",
                visible=True,
                legendgroup=str(item_id),
                line=dict(dash='dash'),
                hovertemplate="<b>Predictions</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>",
                showlegend=show_legend
            ))

            # Add prediction interval (90% confidence band)
            fig.add_trace(go.Scatter(
                x=list(pred_data["timestamp"]) + list(pred_data["timestamp"][::-1]),
                y=list(pred_data["0.9"]) + list(pred_data["0.1"][::-1]),
                fill='toself',
                fillcolor='rgba(230, 200, 255, 0.3)',
                line=dict(color='rgba(255,255,255,0)'),
                name='90% Prediction Interval',
                legendgroup=str(item_id),
                showlegend=show_legend,
                hoverinfo='skip'
            ))
            show_legend = False

fig.update_layout(
    title="Ground Truth vs Predictions - RUL Degradation Level",
    xaxis_title="Time",
    yaxis_title="Degradation Level (Target)",
    width=1400,
    height=600,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

In [53]:
from torch.optim.lr_scheduler import CosineAnnealingLR

pipeline: Chronos2Pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# Prepare training inputs for fine-tuning
train_inputs = []
for item_id in split_train_data["item_id"].unique():
    machine_data = split_train_data[split_train_data["item_id"] == item_id].sort_values("timestamp")
    
    target = machine_data["target"].values
    past_covariates = {
        "brake_temperature": machine_data["brake_temperature"].values,
        "vibration_level": machine_data["vibration_level"].values
    }
    
    train_inputs.append({
        "target": target,
        "past_covariates": past_covariates,
        "future_covariates": {}
    })

# Fine-tune the pipeline with cosine annealing learning rate scheduler

pipeline = pipeline.fit(
    train_inputs,
    num_steps=800,
    learning_rate=1e-5,
    finetune_mode="full",
    prediction_length=prediction_length,
    batch_size=32,
)

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
100,0.581800
200,0.519700
300,0.494400
400,0.492100
500,0.500600
600,0.486800
700,0.462500
800,0.442800


In [54]:
# Generate predictions with covariates
train_pred_df = pipeline.predict_df(
    split_train_data_for_prediction,
    prediction_length=prediction_length,
)

# Compute weighted quantile loss
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
quantile_cols = [str(q) for q in quantiles]

# Merge predictions with ground truth
merged_df = train_pred_df.merge(
    split_train_data[['item_id', 'timestamp', 'target']],
    on=['item_id', 'timestamp'],
    how='inner'
)

# Calculate quantile loss for each quantile
quantile_losses = {}
for q, col in zip(quantiles, quantile_cols):
    # Quantile loss: (q - 1) * (y - pred) if y < pred else q * (y - pred)
    errors = merged_df['target'] - merged_df[col]
    loss = np.where(errors >= 0, q * errors, (q - 1) * errors).mean()
    quantile_losses[col] = loss

# Calculate overall weighted quantile loss
weighted_loss = np.mean(list(quantile_losses.values()))

# Save results
full_fine_tuning_results = {
    'quantile_losses': quantile_losses,
    'weighted_quantile_loss': weighted_loss,
    'merged_predictions': merged_df
}

In [55]:
# Prepare data for visualization
fig = go.Figure()

# Get unique item_ids
item_ids = train_data["item_id"].unique()

for item_id in item_ids:
    show_legend = True
    # Get ground truth data
    truth_data = train_data[train_data["item_id"] == item_id]

    if len(truth_data) > 0:
        # Add ground truth line
        fig.add_trace(go.Scatter(
            x=truth_data["timestamp"],
            y=truth_data["target"],
            mode='lines',
            name=f"Truth {item_id}",
            visible=True,
            legendgroup=str(item_id),
            hovertemplate="<b>Ground Truth</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>"
        ))

    
    # Get prediction data
    pred_item_ids = train_pred_df["item_id"].unique()
    for pred_item_id in pred_item_ids:
        if pred_item_id.startswith(str(item_id) + "_"):
            pred_data = train_pred_df[train_pred_df["item_id"] == pred_item_id]
    
        if len(pred_data) > 0:
        
            # Add prediction line
            fig.add_trace(go.Scatter(
                x=pred_data["timestamp"],
                y=pred_data["predictions"],
                mode='lines',
                name=f"Predictions {item_id}",
                visible=True,
                legendgroup=str(item_id),
                line=dict(dash='dash'),
                hovertemplate="<b>Predictions</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>",
                showlegend=show_legend
            ))

            # Add prediction interval (90% confidence band)
            fig.add_trace(go.Scatter(
                x=list(pred_data["timestamp"]) + list(pred_data["timestamp"][::-1]),
                y=list(pred_data["0.9"]) + list(pred_data["0.1"][::-1]),
                fill='toself',
                fillcolor='rgba(230, 200, 255, 0.3)',
                line=dict(color='rgba(255,255,255,0)'),
                name='90% Prediction Interval',
                legendgroup=str(item_id),
                showlegend=show_legend,
                hoverinfo='skip'
            ))
            show_legend = False

fig.update_layout(
    title="Ground Truth vs Predictions - RUL Degradation Level",
    xaxis_title="Time",
    yaxis_title="Degradation Level (Target)",
    width=1400,
    height=600,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

In [60]:
pipeline: Chronos2Pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# Prepare training inputs for fine-tuning
train_inputs = []
for item_id in split_train_data["item_id"].unique():
    machine_data = split_train_data[split_train_data["item_id"] == item_id].sort_values("timestamp")
    
    target = machine_data["target"].values
    past_covariates = {
        "brake_temperature": machine_data["brake_temperature"].values,
        "vibration_level": machine_data["vibration_level"].values
    }
    
    train_inputs.append({
        "target": target,
        "past_covariates": past_covariates,
        "future_covariates": {}
    })

# Fine-tune the pipeline
pipeline = pipeline.fit(
    train_inputs,
    num_steps=1000,
    learning_rate=1e-4,
    finetune_mode="lora",
    prediction_length=prediction_length,
    batch_size=32,
)

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
100,0.672800
200,0.537800
300,0.519000
400,0.520900
500,0.529400
600,0.515000
700,0.498300
800,0.481400
900,0.504700
1000,0.512500


In [61]:
# Generate predictions with covariates
train_pred_df = pipeline.predict_df(
    split_train_data_for_prediction,
    prediction_length=prediction_length,
)

# Compute weighted quantile loss
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
quantile_cols = [str(q) for q in quantiles]

# Merge predictions with ground truth
merged_df = train_pred_df.merge(
    split_train_data[['item_id', 'timestamp', 'target']],
    on=['item_id', 'timestamp'],
    how='inner'
)

# Calculate quantile loss for each quantile
quantile_losses = {}
for q, col in zip(quantiles, quantile_cols):
    # Quantile loss: (q - 1) * (y - pred) if y < pred else q * (y - pred)
    errors = merged_df['target'] - merged_df[col]
    loss = np.where(errors >= 0, q * errors, (q - 1) * errors).mean()
    quantile_losses[col] = loss

# Calculate overall weighted quantile loss
weighted_loss = np.mean(list(quantile_losses.values()))

# Save results
lora_fine_tuning_results = {
    'quantile_losses': quantile_losses,
    'weighted_quantile_loss': weighted_loss,
    'merged_predictions': merged_df
}

In [62]:
# Prepare data for visualization
fig = go.Figure()

# Get unique item_ids
item_ids = train_data["item_id"].unique()

for item_id in item_ids:
    show_legend = True
    # Get ground truth data
    truth_data = train_data[train_data["item_id"] == item_id]

    if len(truth_data) > 0:
        # Add ground truth line
        fig.add_trace(go.Scatter(
            x=truth_data["timestamp"],
            y=truth_data["target"],
            mode='lines',
            name=f"Truth {item_id}",
            visible=True,
            legendgroup=str(item_id),
            hovertemplate="<b>Ground Truth</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>"
        ))

    
    # Get prediction data
    pred_item_ids = train_pred_df["item_id"].unique()
    for pred_item_id in pred_item_ids:
        if pred_item_id.startswith(str(item_id) + "_"):
            pred_data = train_pred_df[train_pred_df["item_id"] == pred_item_id]
    
        if len(pred_data) > 0:
        
            # Add prediction line
            fig.add_trace(go.Scatter(
                x=pred_data["timestamp"],
                y=pred_data["predictions"],
                mode='lines',
                name=f"Predictions {item_id}",
                visible=True,
                legendgroup=str(item_id),
                line=dict(dash='dash'),
                hovertemplate="<b>Predictions</b><br>Time: %{x}<br>Target: %{y:.4f}<extra></extra>",
                showlegend=show_legend
            ))

            # Add prediction interval (90% confidence band)
            fig.add_trace(go.Scatter(
                x=list(pred_data["timestamp"]) + list(pred_data["timestamp"][::-1]),
                y=list(pred_data["0.9"]) + list(pred_data["0.1"][::-1]),
                fill='toself',
                fillcolor='rgba(230, 200, 255, 0.3)',
                line=dict(color='rgba(255,255,255,0)'),
                name='90% Prediction Interval',
                legendgroup=str(item_id),
                showlegend=show_legend,
                hoverinfo='skip'
            ))
            show_legend = False

fig.update_layout(
    title="Ground Truth vs Predictions - RUL Degradation Level",
    xaxis_title="Time",
    yaxis_title="Degradation Level (Target)",
    width=1400,
    height=600,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

In [63]:
# Create a comparison dataframe of the three approaches
comparison_results = pd.DataFrame({
    'Approach': ['Zero Shot', 'Full Fine-tuning', 'LoRA Fine-tuning'],
    'Weighted Quantile Loss': [
        zero_shot_results['weighted_quantile_loss'],
        full_fine_tuning_results['weighted_quantile_loss'],
        lora_fine_tuning_results['weighted_quantile_loss']
    ]
})

print("Model Performance Comparison:")
print(comparison_results)
print()

# Create a detailed comparison of quantile losses
quantile_cols = ['0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9']
detailed_comparison = pd.DataFrame({
    'Quantile': quantile_cols,
    'Zero Shot': [zero_shot_results['quantile_losses'][q] for q in quantile_cols],
    'Full Fine-tuning': [full_fine_tuning_results['quantile_losses'][q] for q in quantile_cols],
    'LoRA Fine-tuning': [lora_fine_tuning_results['quantile_losses'][q] for q in quantile_cols]
})

print("Detailed Quantile Loss Comparison:")
print(detailed_comparison.to_string(index=False))
print()

# Visualize the comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    x=comparison_results['Approach'],
    y=comparison_results['Weighted Quantile Loss'],
    name='Weighted Quantile Loss',
    marker=dict(color=['#1f77b4', '#ff7f0e', '#2ca02c'])
))

fig.update_layout(
    title='Model Performance Comparison: Weighted Quantile Loss',
    xaxis_title='Approach',
    yaxis_title='Weighted Quantile Loss',
    height=400,
    width=800,
    template='plotly_white'
)
fig.show()

# Visualize quantile-wise comparison
fig = go.Figure()

for approach in ['Zero Shot', 'Full Fine-tuning', 'LoRA Fine-tuning']:
    fig.add_trace(go.Scatter(
        x=detailed_comparison['Quantile'],
        y=detailed_comparison[approach],
        mode='lines+markers',
        name=approach
    ))

fig.update_layout(
    title='Quantile Loss Comparison Across Models',
    xaxis_title='Quantile',
    yaxis_title='Quantile Loss',
    height=500,
    width=1000,
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

Model Performance Comparison:
           Approach  Weighted Quantile Loss
0         Zero Shot                0.379047
1  Full Fine-tuning                0.202453
2  LoRA Fine-tuning                0.234404

Detailed Quantile Loss Comparison:
Quantile  Zero Shot  Full Fine-tuning  LoRA Fine-tuning
     0.1   0.143280          0.107761          0.112912
     0.2   0.245882          0.175874          0.190779
     0.3   0.328304          0.222458          0.246000
     0.4   0.395159          0.249635          0.280394
     0.5   0.445170          0.260732          0.301351
     0.6   0.477796          0.253150          0.300514
     0.7   0.494295          0.234562          0.280511
     0.8   0.476140          0.193572          0.237555
     0.9   0.405396          0.124335          0.159623

